# CS 5542 — Lab 3: Multimodal RAG Systems & Retrieval Evaluation  
**Text + Images/PDFs (runs offline by default; optional LLM API hook)**

This notebook is a **student-ready, simplified, and fully runnable** lab workflow for **multimodal retrieval-augmented generation (RAG)**:
- ingest **PDF text** + **image captions/filenames**
- retrieve evidence with a lightweight baseline (TF‑IDF)
- build a **context block** for answering
- evaluate retrieval quality (Precision@5, Recall@10)
- run an **ablation study** (REQUIRED)

> ✅ **Important:** The code is optimized for **clarity + reproducibility for students** (minimal dependencies, no keys required).  
> It is not the “fastest possible” or “best-performing” RAG system — but it is a correct baseline that you can extend.

---

## Student Tasks (what you must do)
1. **Ingest** PDFs + images from `project_data_mm/` (or use the provided sample package).  
2. Implement / experiment with **chunking strategies** (page-based vs fixed-size).  
3. Compare retrieval methods (at least):  
   - **Sparse** (TF‑IDF / BM25-style)  
   - **Dense** (optional: embeddings)  
   - **Hybrid** (score fusion with `alpha`)  
   - **Hybrid + rerank** (optional: reranker / LLM rerank)  
4. Build a **multimodal context** that includes **evidence items** (text + images).  
5. Produce the required **results table**:

`Query × Method × Precision@5 × Recall@10 × Faithfulness`

---

## Expected Outputs (what graders look for)
- Printed ingestion counts (how many PDF pages/chunks, how many images)
- A retrieval demo showing **top‑k evidence** for a query
- Evaluation metrics per method (P@5, R@10)
- An ablation section with a small comparison table + short explanation


## Key Parameters You Can Tune (and what they do)

These parameters control retrieval + context building. **Students should change them and report what happens.**

- **`TOP_K_TEXT`**: how many text chunks to consider as candidates.  
  - Larger → more recall, but more noise (lower precision).
- **`TOP_K_IMAGES`**: how many image items to consider as candidates.  
  - Larger → more multimodal evidence, but can add irrelevant images.
- **`TOP_K_EVIDENCE`**: how many total evidence items (text+image) go into the final context.  
  - Larger → longer context; may dilute answer quality.
- **`ALPHA`** *(0 → 1)*: **fusion weight** when mixing text vs image evidence.  
  - `ALPHA = 1.0` → text dominates  
  - `ALPHA = 0.0` → images dominate  
  - typical starting point: `0.5`
- **`CHUNK_SIZE`** (fixed-size chunking): characters per chunk (baseline).  
  - Smaller → more granular retrieval (often higher precision)  
  - Larger → fewer chunks (often higher recall but less specific)
- **`CHUNK_OVERLAP`**: overlap between chunks to avoid cutting important info.  
  - Too high → redundant chunks; too low → missing context boundaries

### What to try (recommended student experiments)
- Keep everything fixed, vary **`ALPHA`**: 0.2, 0.5, 0.8  
- Vary **`TOP_K_TEXT`**: 2, 5, 10  
- Compare **page-based** vs **fixed-size** chunking (required ablation)


## 0) Student Info (Fill in)
- Name: Blake Simpson
- UMKC ID: 16284057
- Course/Section: COMP-SCI 5542 - 0001


## 1) Setup (student-friendly baseline)

This lab starter is designed to be **easy to run** and **easy to modify**:
- **PyMuPDF (`fitz`)** for PDF text extraction
- **scikit-learn** for TF‑IDF retrieval (strong sparse baseline)
- **Pillow** for basic image IO
- Optional: connect an **LLM API** for answer generation (not required to run retrieval + eval)

### Student guideline
- First make sure **retrieval + metrics** run end-to-end.
- Then iterate: chunking → retrieval method → fusion (`ALPHA`) → rerank → faithfulness.

> If you have API keys (e.g., Gemini / OpenAI / etc.), you can plug them into the optional LLM hook later —  
> but your retrieval evaluation should work **without** any external keys.


In [1]:
# Imports
import os, re, glob, json, math
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import numpy as np
import pandas as pd

!pip install PyMuPDF
import fitz  # PyMuPDF
from PIL import Image, ImageDraw, ImageFont

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 67.6 MB/s eta 0:00:00:00:0100:01


In [2]:
# =========================
# Lab Configuration (EDIT ME)
# =========================
# Students: try changing these and observe how retrieval metrics change.

DATA_DIR = "project_data_mm"   # folder containing pdfs/ and figures/
PDF_DIR  = os.path.join(DATA_DIR, "pdfs")
IMG_DIR  = os.path.join(DATA_DIR, "figures")

# Retrieval knobs
TOP_K_TEXT     = 5    # candidate text chunks
TOP_K_IMAGES   = 3    # candidate images (based on captions/filenames)
TOP_K_EVIDENCE = 8    # final evidence items used in the context

# Fusion knob (text vs images)
ALPHA = 0.5  # 0.0 = images dominate, 1.0 = text dominates

# Chunking knobs (for fixed-size chunking ablation)
CHUNK_SIZE    = 900   # characters per chunk
CHUNK_OVERLAP = 150   # overlap characters

# Reproducibility
RANDOM_SEED = 0


## 2) Data folder
Expected structure:
```
project_data_mm/
  pdfs/
    hallucination_mitigation.pdf
    production_rag_deployment.pdf
    rag_architectures_patterns.pdf
    rag_history_and_evolution.pdf
    rag_vs_finetuning.pdf
  figures/
    hallucination_mitigation/
      Types_Of_Hallucination.png
      ...
    rag_vs_finetuning/
      Benefit_Of_RAG_Versus_Finetuning.png
      ...
```

If the folder is missing, we will generate **sample PDFs and images** automatically so you can run and verify the pipeline end-to-end.


In [3]:
# === COLAB SETUP: Clone repo and copy project data ===
# Run this cell if you're in Google Colab to get your project data

!git clone https://github.com/EXC3ll3NTrhyTHM/BigDataAnalytics.git

# Copy the project_data_mm folder (with pdfs/ and figures/)
import shutil, os

src_folder = "BigDataAnalytics/Week 3/project_data_mm"
dst_folder = "project_data_mm"

if os.path.exists(src_folder):
    if os.path.exists(dst_folder):
        shutil.rmtree(dst_folder)
    shutil.copytree(src_folder, dst_folder)
    print(f"✅ Copied {src_folder} to {dst_folder}")
else:
    print(f"⚠️ Source folder not found: {src_folder}")

# Verify the structure
import glob
pdfs = glob.glob("project_data_mm/pdfs/*.pdf")
imgs = glob.glob("project_data_mm/figures/**/*.png", recursive=True)
print(f"Found {len(pdfs)} PDFs and {len(imgs)} images")

Cloning into 'BigDataAnalytics'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 108 (delta 18), reused 99 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 6.54 MiB | 12.76 MiB/s, done.
Resolving deltas: 100% (18/18), done.
✅ Copied BigDataAnalytics/Week 3/project_data_mm to project_data_mm
Found 5 PDFs and 45 images


In [4]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.9 MB/s eta 0:00:0000:01


In [5]:
# Data paths
DATA_DIR = "project_data_mm"
FIG_DIR = os.path.join(DATA_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def _write_sample_pdf(pdf_path: str, title: str, paragraphs: List[str]) -> None:
    """Create a simple multi-page PDF with ReportLab."""
    from reportlab.lib.pagesizes import letter
    from reportlab.pdfgen import canvas

    c = canvas.Canvas(pdf_path, pagesize=letter)
    width, height = letter
    y = height - 72

    c.setFont("Helvetica-Bold", 16)
    c.drawString(72, y, title)
    y -= 36
    c.setFont("Helvetica", 11)

    for p in paragraphs:
        # naive line wrapping
        words = p.split()
        line = ""
        for w in words:
            if len(line) + len(w) + 1 > 95:
                c.drawString(72, y, line)
                y -= 14
                line = w
                if y < 72:
                    c.showPage()
                    y = height - 72
                    c.setFont("Helvetica", 11)
            else:
                line = (line + " " + w).strip()
        if line:
            c.drawString(72, y, line)
            y -= 18

        if y < 72:
            c.showPage()
            y = height - 72
            c.setFont("Helvetica", 11)

    c.save()

def _write_sample_image(img_path: str, label: str, size=(900, 550)) -> None:
    """Create a simple image with a big label. Useful for verifying image ingestion."""
    img = Image.new("RGB", size, (245, 245, 245))
    d = ImageDraw.Draw(img)

    # Try a default font; if not available, PIL will fall back.
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", 48)
    except Exception:
        font = ImageFont.load_default()

    d.rectangle([30, 30, size[0]-30, size[1]-30], outline=(30, 30, 30), width=6)
    d.text((60, 200), label, fill=(20, 20, 20), font=font)
    img.save(img_path)

def ensure_sample_dataset(min_pdfs=2, min_imgs=5) -> None:
    """Create a small dataset if user doesn't have one yet."""
    # Check for PDFs in the pdfs/ subfolder and images recursively in figures/
    pdfs = sorted(glob.glob(os.path.join(DATA_DIR, "pdfs", "*.pdf")))
    imgs = sorted(glob.glob(os.path.join(FIG_DIR, "**/*.*"), recursive=True))
    # Filter out non-image files like .DS_Store
    imgs = [f for f in imgs if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.webp'))]

    if len(pdfs) >= min_pdfs and len(imgs) >= min_imgs:
        print("✅ Found existing dataset:", len(pdfs), "PDFs and", len(imgs), "images.")
        return

    print("⚠️ Dataset incomplete. Creating sample dataset...")

    # PDFs
    pdf1 = os.path.join(DATA_DIR, "sample_doc_rag_basics.pdf")
    pdf2 = os.path.join(DATA_DIR, "sample_doc_multimodal_eval.pdf")

    p1 = [
        "Retrieval-Augmented Generation (RAG) combines a retriever and a generator. The retriever fetches evidence chunks from documents.",
        "A common baseline is TF-IDF retrieval. Another baseline is BM25, which uses term frequency and inverse document frequency.",
        "Good RAG answers should be grounded in the retrieved evidence and should not hallucinate facts that are not supported.",
        "When evidence is missing, the system should say 'I don't know' or request more context.",
    ]
    p2 = [
        "Multimodal RAG includes both text (PDF pages) and images (figures). A simple approach is to attach relevant figures as evidence.",
        "Evaluation can include retrieval metrics such as Precision@k and Recall@k, plus qualitative checks for faithfulness.",
        "Ablation studies vary the chunking strategy, retriever type, or the number of retrieved items.",
        "Rubrics help define what counts as relevant evidence for each query.",
    ]

    _write_sample_pdf(pdf1, "Sample Doc 1: RAG Basics", p1)
    _write_sample_pdf(pdf2, "Sample Doc 2: Multimodal RAG + Evaluation", p2)

    # Images (named so text-based retrieval can match them)
    labels = [
        "figure_rag_pipeline",
        "figure_tfidf_retrieval",
        "figure_bm25_baseline",
        "figure_precision_recall",
        "figure_ablation_study",
    ]
    for lab in labels:
        _write_sample_image(os.path.join(FIG_DIR, f"{lab}.png"), lab)

    print("✅ Sample dataset created.")

ensure_sample_dataset()

pdfs = sorted(glob.glob(os.path.join(DATA_DIR, "pdfs", "*.pdf")))
imgs = sorted(glob.glob(os.path.join(FIG_DIR, "**/*.*"), recursive=True))
# Filter out non-image files like .DS_Store
imgs = [f for f in imgs if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.webp'))]

print("PDFs:", len(pdfs), pdfs)
print("Images:", len(imgs), imgs)


✅ Found existing dataset: 5 PDFs and 45 images.
PDFs: 5 ['project_data_mm/pdfs/hallucination_mitigation.pdf', 'project_data_mm/pdfs/production_rag_deployment.pdf', 'project_data_mm/pdfs/rag_architectures_patterns.pdf', 'project_data_mm/pdfs/rag_history_and_evolution.pdf', 'project_data_mm/pdfs/rag_vs_finetuning.pdf']
Images: 45 ['project_data_mm/figures/hallucination_mitigation/Architectural_Approaches.png', 'project_data_mm/figures/hallucination_mitigation/Emerging_Research_Directions.png', 'project_data_mm/figures/hallucination_mitigation/Hallucination_Mind_Map.png', 'project_data_mm/figures/hallucination_mitigation/Measurement_Approaches_For_Hallucinations.png', 'project_data_mm/figures/hallucination_mitigation/Prompt_Engineering_Techniques.png', 'project_data_mm/figures/hallucination_mitigation/Retrieval_Prevention_Strategies.png', 'project_data_mm/figures/hallucination_mitigation/Types_Of_Hallucination.png', 'project_data_mm/figures/hallucination_mitigation/Verification_Approach_T

## 3) Define your 3 queries + rubrics
**Guideline:** write queries that can be answered using your PDFs/images.

Rubric format below is **simple and runnable**:
- `must_have_keywords`: words/phrases that should appear in relevant evidence
- `optional_keywords`: nice-to-have

Later, retrieval metrics will treat an evidence chunk as relevant if it contains at least one `must_have_keywords` item.


In [6]:
QUERIES = [
    {
        "id": "Q1",
        "question": "What are the main types of hallucination in RAG systems and what causes them?",
        "rubric": {
            "must_have_keywords": ["intrinsic", "extrinsic", "hallucination", "retrieval"],
            "optional_keywords": ["contradicts", "source", "training", "attention", "probability"]
        }
    },
    {
        "id": "Q2",
        "question": "When should I choose RAG over fine-tuning for my LLM application?",
        "rubric": {
            "must_have_keywords": ["rag", "fine-tuning", "knowledge", "cost"],
            "optional_keywords": ["scalability", "latency", "hybrid", "attribution", "domain"]
        }
    },
    {
        "id": "Q3",
        "question": "How do I improve my RAG system?",
        "rubric": {
            "must_have_keywords": ["retrieval", "quality", "architecture"],
            "optional_keywords": ["hybrid", "reranking", "caching", "monitoring", "chunking", "hallucination"]
        }
    },
]


## 4) Ingestion
We extract:
- **PDF per-page text** as `TextChunk`
- **Image metadata** as `ImageItem` (caption = filename without extension)

> This is intentionally lightweight so it runs without downloading large embedding models.


In [9]:
@dataclass
class TextChunk:
    chunk_id: str
    doc_id: str
    page_num: int
    text: str

@dataclass
class ImageItem:
    item_id: str
    path: str
    caption: str  # simple text to make image retrieval runnable

def clean_text(s: str) -> str:
    s = s or ""
    s = re.sub(r"\s+", " ", s).strip()
    return s

def extract_pdf_pages(pdf_path: str) -> List[TextChunk]:
    doc_id = os.path.basename(pdf_path)
    doc = fitz.open(pdf_path)
    out: List[TextChunk] = []
    for i in range(len(doc)):
        page = doc.load_page(i)
        text = clean_text(page.get_text("text"))
        if text:
            out.append(TextChunk(
                chunk_id=f"{doc_id}::p{i+1}",
                doc_id=doc_id,
                page_num=i+1,
                text=text
            ))
    return out

def load_images(fig_dir: str) -> List[ImageItem]:
    items: List[ImageItem] = []
    # Search recursively for images in all subfolders
    for p in sorted(glob.glob(os.path.join(fig_dir, "**/*.*"), recursive=True)):
        # Filter to only image files
        if not p.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.webp')):
            continue
        base = os.path.basename(p)
        caption = os.path.splitext(base)[0].replace("_", " ")
        items.append(ImageItem(item_id=base, path=p, caption=caption))
    return items

# Run ingestion
page_chunks: List[TextChunk] = []
for p in pdfs:
    page_chunks.extend(extract_pdf_pages(p))

image_items = load_images(FIG_DIR)

print("Total text chunks:", len(page_chunks))
print("Total images:", len(image_items))
print("Sample text chunk:", page_chunks[0].chunk_id, page_chunks[0].text[:180])
print("Sample image item:", image_items[0])


Total text chunks: 64
Total images: 45
Sample text chunk: hallucination_mitigation.pdf::p1 Hallucination Mitigation Strategies in RAG Systems Understanding Hallucination in Language Models Hallucination refers to the phenomenon where language models generate content that
Sample image item: ImageItem(item_id='Architectural_Approaches.png', path='project_data_mm/figures/hallucination_mitigation/Architectural_Approaches.png', caption='Architectural Approaches')


## 5) Retrieval (TF‑IDF)
We build two TF‑IDF indexes:
- One over **PDF text chunks**
- One over **image captions**

Retrieval returns the top‑k results with similarity scores.


In [10]:
def build_tfidf_index_text(chunks: List[TextChunk]):
    corpus = [c.text for c in chunks]
    vec = TfidfVectorizer(lowercase=True, stop_words="english")
    X = vec.fit_transform(corpus)
    X = normalize(X)
    return vec, X

def build_tfidf_index_images(items: List[ImageItem]):
    corpus = [it.caption for it in items]
    vec = TfidfVectorizer(lowercase=True, stop_words="english")
    X = vec.fit_transform(corpus)
    X = normalize(X)
    return vec, X

text_vec, text_X = build_tfidf_index_text(page_chunks)
img_vec, img_X = build_tfidf_index_images(image_items)

def tfidf_retrieve(query: str, vec: TfidfVectorizer, X, top_k: int = 5):
    q = vec.transform([query])
    q = normalize(q)
    scores = (X @ q.T).toarray().ravel()
    idx = np.argsort(-scores)[:top_k]
    return [(int(i), float(scores[i])) for i in idx]

print("✅ Indexes built.")


✅ Indexes built.


## 6) Build evidence context
We assemble a compact context string + list of image paths.

**Guidelines for good context:**
- Keep snippets short (100–300 chars)
- Always include chunk IDs so you can cite evidence
- Attach images that are likely relevant


In [11]:
def _normalize_scores(pairs):
    """Min-max normalize a list of (idx, score) to [0,1].
    If all scores equal, returns 1.0 for each item (so ordering stays stable).
    """
    if not pairs:
        return []
    scores = [s for _, s in pairs]
    lo, hi = min(scores), max(scores)
    if abs(hi - lo) < 1e-12:
        return [(i, 1.0) for i, _ in pairs]
    return [(i, (s - lo) / (hi - lo)) for i, s in pairs]


def build_context(
    question: str,
    top_k_text: int = TOP_K_TEXT,
    top_k_images: int = TOP_K_IMAGES,
    top_k_evidence: int = TOP_K_EVIDENCE,
    alpha: float = ALPHA,
) -> Dict[str, Any]:
    """Build a multimodal context block for the question.

    Students:
    - `top_k_text` / `top_k_images` control *candidate retrieval* per modality.
    - `top_k_evidence` controls the *final context size*.
    - `alpha` controls fusion: higher = prefer text evidence, lower = prefer images.

    This function returns:
    - `context`: a text block with the selected evidence (what you pass to an LLM)
    - `image_paths`: paths of images selected as evidence
    - `evidence`: structured evidence list (recommended for your report)
    """
    # 1) Retrieve candidates from each modality
    text_hits = tfidf_retrieve(question, text_vec, text_X, top_k=top_k_text)   # [(idx, score), ...]
    img_hits  = tfidf_retrieve(question, img_vec,  img_X,  top_k=top_k_images)

    # 2) Normalize scores per modality and fuse with ALPHA
    text_norm = _normalize_scores(text_hits)
    img_norm  = _normalize_scores(img_hits)

    fused = []
    for idx, s in text_norm:
        ch = page_chunks[idx]
        fused.append({
            "modality": "text",
            "id": ch.chunk_id,
            "raw_score": float(dict(text_hits).get(idx, 0.0)),
            "fused_score": float(alpha * s),
            "text": ch.text,
            "path": None,
        })

    for idx, s in img_norm:
        it = image_items[idx]
        fused.append({
            "modality": "image",
            "id": it.item_id,
            "raw_score": float(dict(img_hits).get(idx, 0.0)),
            "fused_score": float((1.0 - alpha) * s),
            "text": it.caption,     # we retrieve on caption/filename text
            "path": it.path,
        })

    # 3) Pick top fused evidence
    fused = sorted(fused, key=lambda d: d["fused_score"], reverse=True)[:top_k_evidence]

    # 4) Build the context string (what you feed into a generator/LLM)
    ctx_lines = []
    image_paths = []
    for ev in fused:
        if ev["modality"] == "text":
            snippet = (ev["text"] or "")[:260].replace("\n", " ")
            ctx_lines.append(f"[TEXT | {ev['id']} | fused={ev['fused_score']:.3f}] {snippet}")
        else:
            ctx_lines.append(f"[IMAGE | {ev['id']} | fused={ev['fused_score']:.3f}] caption={ev['text']}")
            image_paths.append(ev["path"])

    return {
        "question": question,
        "context": "\n".join(ctx_lines),
        "image_paths": image_paths,
        "text_hits": text_hits,
        "img_hits": img_hits,
        "evidence": fused,
        "alpha": alpha,
        "top_k_text": top_k_text,
        "top_k_images": top_k_images,
        "top_k_evidence": top_k_evidence,
    }


# --- Demo: what retrieval returns for one query ---
ctx_demo = build_context(QUERIES[0]["question"])
print(ctx_demo["context"])
print("Images:", ctx_demo["image_paths"])
print("Fusion alpha:", ctx_demo["alpha"])


[TEXT | hallucination_mitigation.pdf::p1 | fused=0.500] Hallucination Mitigation Strategies in RAG Systems Understanding Hallucination in Language Models Hallucination refers to the phenomenon where language models generate content that is factually incorrect, nonsensical, or unfaithful to provided source material.
[IMAGE | Types_Of_Hallucination.png | fused=0.500] caption=Types Of Hallucination
[TEXT | hallucination_mitigation.pdf::p2 | fused=0.147] Retrieval-Induced Hallucination Emerges from poor retrieval quality. If retrieved documents are irrelevant or only tangentially related to the query, the model may hallucinate to produce a seemingly responsive answer. Compositional Hallucination Occurs when co
[TEXT | rag_architectures_patterns.pdf::p2 | fused=0.060] Naive RAG vs Advanced RAG Limitations Benefits ✅ Advanced RAG Pre-Retrieval Query rewriting & expansion Hybrid Search BM25 + Dense embeddings Re-Ranking Cross-encoder scoring Compression Remove redundancy ❌ Naive RAG Single re

## 7) “Generator” (simple, offline)
To keep this notebook runnable anywhere, we implement a **lightweight extractive generator**:
- It returns the top evidence lines
- In your real submission, you can replace this with an LLM call (HF local model or an API)

**Key rule:** the answer must stay consistent with evidence.


In [12]:
def simple_extractive_answer(question: str, context: str) -> str:
    lines = context.splitlines()
    if not lines:
        return "I don't know (no evidence retrieved)."
    # Return top 2 evidence lines as a "grounded" answer
    return (
        f"Question: {question}\n\n"
        "Grounded answer (extractive):\n"
        + "\n".join(lines[:2])
    )

def run_query(qobj, top_k_text=TOP_K_TEXT, top_k_images=TOP_K_IMAGES, top_k_evidence=TOP_K_EVIDENCE, alpha=ALPHA) -> Dict[str, Any]:
    question = qobj["question"]
    ctx = build_context(question, top_k_text=top_k_text, top_k_images=top_k_images, top_k_evidence=top_k_evidence, alpha=alpha)
    answer = simple_extractive_answer(question, ctx["context"])
    return {
        "id": qobj["id"],
        "question": question,
        "answer": answer,
        "context": ctx["context"],
        "image_paths": ctx["image_paths"],
        "text_hits": ctx["text_hits"],
        "img_hits": ctx["img_hits"],
    }

results = [run_query(q) for q in QUERIES]
for r in results:
    print("\n" + "="*80)
    print(r["id"], r["question"])
    print(r["answer"][:500])
    print("Images:", [os.path.basename(p) for p in r["image_paths"]])



Q1 What are the main types of hallucination in RAG systems and what causes them?
Question: What are the main types of hallucination in RAG systems and what causes them?

Grounded answer (extractive):
[TEXT | hallucination_mitigation.pdf::p1 | fused=0.500] Hallucination Mitigation Strategies in RAG Systems Understanding Hallucination in Language Models Hallucination refers to the phenomenon where language models generate content that is factually incorrect, nonsensical, or unfaithful to provided source material.
[IMAGE | Types_Of_Hallucination.png | fused=0.500] caption=Types
Images: ['Types_Of_Hallucination.png', 'Accuracy_Hallucination_Comparison.png', 'Hallucination_Mind_Map.png']

Q2 When should I choose RAG over fine-tuning for my LLM application?
Question: When should I choose RAG over fine-tuning for my LLM application?

Grounded answer (extractive):
[TEXT | rag_vs_finetuning.pdf::p10 | fused=0.500] Enterprise Knowledge Base Chatbot: Choose RAG. Documents change frequently, user

## 8) Retrieval Evaluation (Precision@k / Recall@k)
We treat a text chunk as **relevant** for a query if it contains at least one `must_have_keywords` term.



In [13]:
def is_relevant_text(chunk_text: str, rubric: Dict[str, Any]) -> bool:
    text = chunk_text.lower()
    must = [k.lower() for k in rubric.get("must_have_keywords", [])]
    return any(k in text for k in must)

def precision_at_k(relevances: List[bool], k: int) -> float:
    k = min(k, len(relevances))
    if k == 0:
        return 0.0
    return sum(relevances[:k]) / k

def recall_at_k(relevances: List[bool], k: int, total_relevant: int) -> float:
    k = min(k, len(relevances))
    if total_relevant == 0:
        return 0.0
    return sum(relevances[:k]) / total_relevant

def eval_retrieval_for_query(qobj, top_k=10) -> Dict[str, Any]:
    question = qobj["question"]
    rubric = qobj["rubric"]

    hits = tfidf_retrieve(question, text_vec, text_X, top_k=top_k)
    rels = []
    for i, score in hits:
        rels.append(is_relevant_text(page_chunks[i].text, rubric))

    # Estimate total relevant in the corpus (for recall)
    total_rel = sum(is_relevant_text(ch.text, rubric) for ch in page_chunks)

    return {
        "id": qobj["id"],
        "P@5": precision_at_k(rels, 5),
        "R@10": recall_at_k(rels, 10, total_rel),
        "total_relevant_chunks": total_rel,
    }

eval_rows = [eval_retrieval_for_query(q) for q in QUERIES]
df_eval = pd.DataFrame(eval_rows)
df_eval


,id,P@5,R@10,total_relevant_chunks
0,Q1,1.0,0.204082,49
1,Q2,1.0,0.185185,54
2,Q3,0.6,0.150943,53


## 9) Ablation Study (REQUIRED)

You must compare **at least**:
- **Chunking A (page-based)** vs **Chunking B (fixed-size)**  
- **Sparse** vs **Dense** vs **Hybrid** vs **Hybrid + Rerank** *(dense/rerank can be optional extensions — but include at least sparse + one fusion variant)*  
- **Text-only RAG** vs **Multimodal RAG** (your context must include evidence items)

**Deliverable:** include a final results table in your README:

`Query × Method × Precision@5 × Recall@10 × Faithfulness`

### Quick ablation ideas
- Vary `TOP_K_TEXT`: 2, 5, 10  
- Vary `ALPHA`: 0.2, 0.5, 0.8  
- Compare page-chunking vs fixed-size (`CHUNK_SIZE` / `CHUNK_OVERLAP`)  


In [18]:
# ============================================================
# ABLATION 1: Page-based vs Fixed-size Chunking
# ============================================================

def fixed_size_chunk(full_text: str, doc_id: str,
                     chunk_size: int = CHUNK_SIZE,
                     overlap: int = CHUNK_OVERLAP) -> List[TextChunk]:
    """Split a document's full text into fixed-size character chunks with overlap."""
    chunks = []
    start = 0
    idx = 0
    while start < len(full_text):
        end = start + chunk_size
        chunk_text = full_text[start:end].strip()
        if chunk_text:
            chunks.append(TextChunk(
                chunk_id=f"{doc_id}::fix{idx}",
                doc_id=doc_id,
                page_num=idx,
                text=chunk_text
            ))
            idx += 1
        start += chunk_size - overlap
    return chunks

# Build fixed-size chunks from all PDFs
fixed_chunks: List[TextChunk] = []
for p in pdfs:
    doc = fitz.open(p)
    full_text = " ".join(clean_text(doc.load_page(i).get_text("text")) for i in range(len(doc)))
    fixed_chunks.extend(fixed_size_chunk(full_text, os.path.basename(p)))

print(f"Page-based chunks: {len(page_chunks)}")
print(f"Fixed-size chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}): {len(fixed_chunks)}")

# Build TF-IDF index for fixed-size chunks
fixed_vec, fixed_X = build_tfidf_index_text(fixed_chunks)

def eval_with_index(qobj, chunks, vec, X, top_k=10):
    """Evaluate retrieval on a specific chunk index."""
    question = qobj["question"]
    rubric = qobj["rubric"]
    hits = tfidf_retrieve(question, vec, X, top_k=top_k)
    rels = [is_relevant_text(chunks[i].text, rubric) for i, _ in hits]
    total_rel = sum(is_relevant_text(ch.text, rubric) for ch in chunks)
    return {
        "P@5": precision_at_k(rels, 5),
        "R@10": recall_at_k(rels, 10, total_rel),
    }

# Compare chunking strategies
chunking_rows = []
for q in QUERIES:
    page_m = eval_with_index(q, page_chunks, text_vec, text_X)
    fixed_m = eval_with_index(q, fixed_chunks, fixed_vec, fixed_X)
    chunking_rows.append({"Query": q["id"], "Chunking": "Page-based", **page_m})
    chunking_rows.append({"Query": q["id"], "Chunking": f"Fixed ({CHUNK_SIZE} chars)", **fixed_m})

df_chunking = pd.DataFrame(chunking_rows)
print("\n=== Ablation 1: Chunking Strategy Comparison ===")
df_chunking

Page-based chunks: 64
Fixed-size chunks (size=900, overlap=150): 88

=== Ablation 1: Chunking Strategy Comparison ===


,Query,Chunking,P@5,R@10
0,Q1,Page-based,1.0,0.204082
1,Q1,Fixed (900 chars),1.0,0.142857
2,Q2,Page-based,1.0,0.185185
3,Q2,Fixed (900 chars),1.0,0.128205
4,Q3,Page-based,0.6,0.150943
5,Q3,Fixed (900 chars),0.8,0.114286


In [19]:
# ============================================================
# ABLATION 2: Text-only vs Multimodal (Alpha Variation)
# ============================================================

def eval_multimodal(qobj, alpha, top_k_text=TOP_K_TEXT,
                    top_k_images=TOP_K_IMAGES,
                    top_k_evidence=TOP_K_EVIDENCE):
    """Evaluate fused multimodal retrieval at a given alpha."""
    ctx = build_context(qobj["question"], top_k_text=top_k_text,
                        top_k_images=top_k_images,
                        top_k_evidence=top_k_evidence, alpha=alpha)
    rubric = qobj["rubric"]

    # Relevance of each evidence item (text chunk or image caption)
    rels = [is_relevant_text(ev["text"], rubric) for ev in ctx["evidence"]]

    # Total relevant across both modalities
    total_rel_text = sum(is_relevant_text(ch.text, rubric) for ch in page_chunks)
    total_rel_img = sum(is_relevant_text(it.caption, rubric) for it in image_items)
    total_rel = total_rel_text + total_rel_img

    n_text = sum(1 for ev in ctx["evidence"] if ev["modality"] == "text")
    n_img = sum(1 for ev in ctx["evidence"] if ev["modality"] == "image")

    return {
        "P@5": precision_at_k(rels, 5),
        "R@10": recall_at_k(rels, 10, total_rel),
        "Text Items": n_text,
        "Image Items": n_img,
    }

alpha_values = [1.0, 0.8, 0.5, 0.2, 0.0]
alpha_labels = {1.0: "Text-only", 0.8: "Text-heavy", 0.5: "Balanced",
                0.2: "Image-heavy", 0.0: "Images-only"}

alpha_rows = []
for q in QUERIES:
    for a_val in alpha_values:
        m = eval_multimodal(q, alpha=a_val)
        alpha_rows.append({
            "Query": q["id"], "Alpha": a_val, "Mode": alpha_labels[a_val], **m
        })

df_alpha = pd.DataFrame(alpha_rows)
print("=== Ablation 2: Multimodal Fusion (Alpha) ===")
df_alpha

=== Ablation 2: Multimodal Fusion (Alpha) ===


,Query,Alpha,Mode,P@5,R@10,Text Items,Image Items
0,Q1,1.0,Text-only,1.0,0.142857,5,3
1,Q1,0.8,Text-heavy,1.0,0.142857,5,3
2,Q1,0.5,Balanced,1.0,0.142857,5,3
3,Q1,0.2,Image-heavy,1.0,0.142857,5,3
4,Q1,0.0,Images-only,1.0,0.142857,5,3
5,Q2,1.0,Text-only,1.0,0.111111,5,3
6,Q2,0.8,Text-heavy,0.8,0.111111,5,3
7,Q2,0.5,Balanced,0.8,0.111111,5,3
8,Q2,0.2,Image-heavy,0.8,0.111111,5,3
9,Q2,0.0,Images-only,0.8,0.111111,5,3


In [20]:
# ============================================================
# FINAL RESULTS TABLE (Required Deliverable)
# Query x Method x Precision@5 x Recall@10 x Faithfulness
# ============================================================

def compute_faithfulness(evidence_texts: List[str], rubric: Dict[str, Any]) -> float:
    """Faithfulness = fraction of must_have_keywords found in the retrieved evidence."""
    must = [k.lower() for k in rubric.get("must_have_keywords", [])]
    if not must:
        return 1.0
    combined = " ".join(t.lower() for t in evidence_texts)
    return sum(1 for k in must if k in combined) / len(must)

def get_text_only_evidence(qobj, chunks, vec, X, top_k=10) -> List[str]:
    """Get evidence texts from text-only retrieval."""
    hits = tfidf_retrieve(qobj["question"], vec, X, top_k=top_k)
    return [chunks[i].text for i, _ in hits]

def get_multimodal_evidence(qobj, alpha) -> List[str]:
    """Get evidence texts from multimodal fusion retrieval."""
    ctx = build_context(qobj["question"], alpha=alpha)
    return [ev["text"] for ev in ctx["evidence"]]

# Build the summary table
summary_rows = []

methods = [
    # (label, chunking, alpha, is_multimodal)
    ("Sparse (TF-IDF) + Page chunks, text-only",     "page",  1.0, False),
    ("Sparse (TF-IDF) + Page chunks, multimodal",    "page",  0.5, True),
    ("Sparse (TF-IDF) + Fixed chunks, text-only",    "fixed", 1.0, False),
    ("Sparse (TF-IDF) + Fixed chunks, multimodal",   "fixed", 0.5, True),
    ("Hybrid fusion (alpha=0.2, image-heavy)",        "page",  0.2, True),
]

for q in QUERIES:
    for method_name, chunk_type, alpha_val, is_mm in methods:
        # Select the right index
        if chunk_type == "page":
            chunks, vec, X = page_chunks, text_vec, text_X
        else:
            chunks, vec, X = fixed_chunks, fixed_vec, fixed_X

        # Retrieval metrics
        if is_mm:
            m = eval_multimodal(q, alpha=alpha_val)
        else:
            m = eval_with_index(q, chunks, vec, X)

        # Faithfulness
        if is_mm:
            ev_texts = get_multimodal_evidence(q, alpha=alpha_val)
        else:
            ev_texts = get_text_only_evidence(q, chunks, vec, X)
        faith = compute_faithfulness(ev_texts, q["rubric"])

        summary_rows.append({
            "Query": q["id"],
            "Method": method_name,
            "P@5": round(m["P@5"], 3),
            "R@10": round(m["R@10"], 3),
            "Faithfulness": round(faith, 3),
        })

df_summary = pd.DataFrame(summary_rows)
print("=== Final Results: Query x Method x P@5 x R@10 x Faithfulness ===\n")
# Display with full width so Method column isn't truncated
with pd.option_context("display.max_colwidth", 60, "display.width", 120):
    print(df_summary.to_string(index=False))

=== Final Results: Query x Method x P@5 x R@10 x Faithfulness ===

Query                                     Method  P@5  R@10  Faithfulness
   Q1   Sparse (TF-IDF) + Page chunks, text-only  1.0 0.204           1.0
   Q1  Sparse (TF-IDF) + Page chunks, multimodal  1.0 0.143           1.0
   Q1  Sparse (TF-IDF) + Fixed chunks, text-only  1.0 0.143           1.0
   Q1 Sparse (TF-IDF) + Fixed chunks, multimodal  1.0 0.143           1.0
   Q1     Hybrid fusion (alpha=0.2, image-heavy)  1.0 0.143           1.0
   Q2   Sparse (TF-IDF) + Page chunks, text-only  1.0 0.185           1.0
   Q2  Sparse (TF-IDF) + Page chunks, multimodal  0.8 0.111           1.0
   Q2  Sparse (TF-IDF) + Fixed chunks, text-only  1.0 0.128           1.0
   Q2 Sparse (TF-IDF) + Fixed chunks, multimodal  0.8 0.111           1.0
   Q2     Hybrid fusion (alpha=0.2, image-heavy)  0.8 0.111           1.0
   Q3   Sparse (TF-IDF) + Page chunks, text-only  0.6 0.151           1.0
   Q3  Sparse (TF-IDF) + Page chunks, multimo

### Ablation Analysis

**Chunking Strategy (Page-based vs Fixed-size):**
- Fixed-size chunking (900 chars, 150 overlap) improved precision for the broadest query — Q3 ("How do I improve my RAG system?") went from **P@5 = 0.6** (page-based) to **P@5 = 0.8** (fixed-size) in text-only mode. Smaller, more uniform chunks isolate specific passages better when the query is vague and many pages are only partially relevant.
- For the more targeted queries (Q1 and Q2), page-based chunking performed equally well or better, since the most relevant pages are already highly focused on the topic.
- Recall was slightly higher with page-based chunking (e.g., Q1: 0.204 vs 0.143) because each page-sized chunk covers more content per hit, capturing more rubric keywords in fewer retrievals.

**Text-only vs Multimodal (Alpha Variation):**
- Text-only retrieval (alpha=1.0) consistently outperformed multimodal variants on both P@5 and R@10. For Q2, P@5 dropped from 1.0 (text-only) to 0.8 (multimodal). For Q3, it dropped from 0.6 to 0.4.
- The reason: image captions are derived from filenames (e.g., "RAG Cost Breakdown") and are too short to reliably contain the rubric's `must_have_keywords`. When images take evidence slots away from text chunks, they count as irrelevant hits under keyword-based evaluation, reducing precision.
- However, multimodal evidence is still valuable in practice — a real LLM generator can interpret figures visually, providing richer answers even if the captions alone don't match keyword rubrics.

**Faithfulness:**
- Faithfulness was **1.0 across all methods and queries**, meaning every configuration retrieved evidence containing all required `must_have_keywords`. This indicates the TF-IDF retriever reliably surfaces chunks covering the core concepts for each query.
- The uniformly high faithfulness also reflects that the rubric keywords (e.g., "hallucination", "retrieval", "rag", "cost") are broad terms that appear frequently across the corpus. More specific or multi-word keywords would create greater differentiation between methods.

**Key Takeaways:**
1. **Chunking strategy matters most for broad queries** — fixed-size chunking improved Q3 precision by 33% over page-based, while targeted queries saw no difference.
2. **Keyword-based evaluation penalizes multimodal retrieval** because short image captions rarely match rubric terms, even when the images themselves are highly relevant.
3. **All methods achieve full faithfulness**, confirming that TF-IDF is a reliable baseline for surfacing topically relevant evidence in this corpus.

## 10) What to submit
1) Your updated dataset (or keep your own)
2) This notebook (with your answers + screenshots/outputs)
3) A short write‑up: retrieval metrics + faithfulness discussion + ablation

**Tip:** If you switch to an LLM, keep the same `build_context()` so the evidence is always visible.
